In [1]:
# ============================================================================
# INPUT CHECK - runs first, before the stage below it.
#
# Every stage resolves its inputs with glob("/kaggle/input/**/<name>") and takes
# the FIRST hit alphabetically. A file attached twice is therefore not an error,
# it is a coin toss - and the order is worse than random: a folder named
# stage1-data-pipeline-OLD sorts BEFORE stage1-data-pipeline, so the copy you
# meant to retire is the one that wins, silently.
#
# This prints what will actually be read, and stops the run if anything is
# attached twice. Everything lives inside one function so it cannot collide
# with a name the stage below uses.
# ============================================================================
def _check_kaggle_inputs():
    import glob, os
    from datetime import datetime

    WANTED = {
        "unified.parquet":                     "Stage 1 corpus        (Stages 2, 4)",
        "splits.json":                         "Stage 1 splits        (Stage 2)",
        "predictions_finetuned.parquet":       "Stage 2 encoder store (Stage 2 resume, 4, 5)",
        "predictions_llm.parquet":             "Stage 3 LLM binary    (Stages 4, 5)",
        "predictions_subtype.parquet":         "Stage 3b raw subtype  (Stage 3b-repair)",
        "predictions_subtype_repaired.parquet": "Stage 3b repaired    (Stages 4, 5)",
        "predictions_llm_repaired.parquet":     "Stage 3b R8 binary   (Stages 4, 5)",
        "tab9_evaluability.csv":               "Stage 4 gate          (Stage 5)",
    }
    # Files THIS stage actually reads. A bad version of one of these is a
    # hard stop; anything else in WANTED is checked for information only.
    CONSUMED = {"predictions_finetuned.parquet",
                "predictions_llm_repaired.parquet",
                "predictions_subtype_repaired.parquet",
                "tab9_evaluability.csv"}

    print("=" * 78)
    print("ATTACHED INPUTS")
    print("=" * 78)
    problems = []
    for fname, used_by in WANTED.items():
        hits = sorted(glob.glob(f"/kaggle/input/**/{fname}", recursive=True))
        print(f"\n{fname}   <- {used_by}")
        if not hits:
            print("   (not attached)")
            continue
        for i, h in enumerate(hits):
            when = datetime.fromtimestamp(os.path.getmtime(h)).strftime("%Y-%m-%d %H:%M")
            mark = "  >> THIS ONE WILL BE USED" if i == 0 else "     ignored"
            print(f"   {h}\n      {os.path.getsize(h):>12,} bytes   {when}{mark}")
        if len(hits) > 1:
            problems.append(f"DUPLICATE: {fname} is attached {len(hits)} times")

    print("\n" + "=" * 78)
    print("ROW COUNTS OF WHAT WILL ACTUALLY BE READ")
    print("=" * 78)
    # A file of the right NAME can still be the wrong VERSION. The row count is
    # what tells them apart. The encoder store has two legitimate sizes
    # depending on where you are in the chain, so it is checked against both.
    EXPECTED = {
        "unified.parquet":                     ({1412}, "1,412"),
        "predictions_finetuned.parquet":       ({16900, 24280},
                                                "16,900 before Stage 2 / 24,280 after it"),
        "predictions_llm.parquet":             ({35049}, "35,049"),
        "predictions_subtype_repaired.parquet": ({16786}, "16,786"),
        "predictions_llm_repaired.parquet":     ({35049}, "35,049"),
    }
    try:
        import pandas as pd
        for fname, (ok_values, note) in EXPECTED.items():
            hits = sorted(glob.glob(f"/kaggle/input/**/{fname}", recursive=True))
            if not hits:
                continue
            n = len(pd.read_parquet(hits[0]))
            # The expected values are the counts of the COMMITTED REFERENCE
            # run. A LARGER prediction store is legitimate after an API top-up
            # (the harnesses resume and add previously rate-limited items); a
            # SMALLER one is an old or partial store and must not be used.
            # unified.parquet must match exactly: requirement ids are
            # positional, so a corpus of any other size silently re-points
            # every stored prediction.
            if n in ok_values:
                flag = "OK"
            elif fname == "unified.parquet":
                flag = "<-- CORPUS SIZE CHANGED: ids are positional, do NOT run"
            elif n > max(ok_values):
                flag = ("larger than the committed reference (expected after an "
                        "API top-up; verify provenance, then update EXPECTED here)")
            else:
                flag = ("<-- SMALLER than the committed reference "
                        "(old/partial store), do not run")
            print(f"  {fname:38s} {n:>7,} rows   expected {note}   {flag}")
            # A '<--' flag on a file THIS stage actually consumes is a hard
            # stop: the old check printed 'do not run' and then ran anyway,
            # which is how hours of compute get spent against a wrong input.
            if flag.startswith("<--") and fname in CONSUMED:
                problems.append(f"BAD INPUT: {fname} - {flag.lstrip('<- ')}")
            if fname == "predictions_finetuned.parquet":
                which = ("the OLD store (correct INPUT for Stage 2 itself)" if n == 16900
                         else "the NEW store (required by Stages 4 and 5)" if n == 24280
                         else "neither size this chain produces")
                print(f"      -> this is {which}")
    except Exception as e:
        print("  (could not read:", e, ")")

    sp = sorted(glob.glob("/kaggle/input/**/splits.json", recursive=True))
    if sp:
        import json
        fams = json.load(open(sp[0]))
        n_folds = sum(len(v) for v in fams.values())
        verdict = "OK" if len(fams) == 14 else "<-- OLD Stage 1 output, re-run Stage 1 first"
        print(f"\n  splits.json: {len(fams)} families, {n_folds} folds   {verdict}")
        if verdict.startswith("<--") and "splits.json" in CONSUMED:
            problems.append("BAD INPUT: splits.json - OLD Stage 1 output "
                            f"({len(fams)} families, expected 14)")

    # Stage 4 and Stage 5 PREFER predictions_llm_repaired over the harness
    # store, and fall back silently when it is absent. Absent means an older
    # stage3b-repair version is attached - one from before R8 existed - and the
    # analysis would then run on the longest-first parse the project moved off.
    # The fallback is by design; being unaware of it is not.
    _raw = sorted(glob.glob("/kaggle/input/**/predictions_llm.parquet", recursive=True))
    _rep = sorted(glob.glob("/kaggle/input/**/predictions_llm_repaired.parquet",
                            recursive=True))
    if _raw and not _rep:
        print("  WARNING: predictions_llm.parquet is attached but")
        print("           predictions_llm_repaired.parquet is NOT.")
        print("           Stages 4 and 5 will silently fall back to the")
        print("           unrepaired binary store. Attach the stage3b-repair")
        print("           version that printed an [R8] line. Harmless for")
        print("           Stage 3b-repair itself, which produces that file.")

    print("\n" + "=" * 78)
    if problems:
        for p in problems:
            print(f"  {p}")
        raise SystemExit("Fix the problems above (detach duplicates / attach "
                         "the right versions), then run again. "
                         "The stage below did NOT run.")
    print("No duplicates. Running the stage now.")
    print("=" * 78 + "\n")


_check_kaggle_inputs()


# =============================================================================
# STAGE 5 - COST EFFICIENCY  (RQ3)
#
# INPUTS (add as Notebook-Output datasets):
#   stage2-finetuned-baselines  -> predictions_finetuned.parquet
#   stage3-llm-harness          -> predictions_llm.parquet
#   stage3b-repair (or topup)   -> predictions_subtype_repaired.parquet
#   stage4-analysis             -> tab9_evaluability.csv   (the reportable gate)
#
# SETTINGS: Accelerator = None, Internet = Off.
#
# Runs no model. Every figure is derived from cost, token and latency columns
# already stored per prediction, so the aggregate can never drift from the
# per-prediction appendix table.
#
# OUTPUTS -> /kaggle/working/
#   cost1_per_model.csv        cost2_pareto.csv        cost3_breakeven.csv
#   cost4_sensitivity.csv      cost5_scaling.csv       stage5_provenance.json
#   fig7_pareto.(png|pdf)      fig8_breakeven.(png|pdf)
#   fig9_cost_per_correct.(png|pdf) fig10_latency.(png|pdf)
#
# RUN ORDER: re-run STAGE 4 FIRST and attach its NEW output here, because the
# reportability gate (tab9_evaluability.csv) must already contain the encoder
# cells or they will be filtered straight back out again.
#
# -----------------------------------------------------------------------------
# REVISION 2  -  what changed and why  (search the file for the [FIX-n] tags)
#
# [FIX-1]  CRITICAL, identical to the Stage 4 defect.  The filter
#              head = store[store.prompt_id.isin([...]) & (store.shot_k == 0)]
#          dropped all 16,900 fine-tuned rows (they carry shot_k = -1 and
#          prompt_id = "supervised"), so cost1 held no encoder, every
#          one_off_training_usd was 0.0, cost3_breakeven.csv came out empty and
#          fig8_breakeven was never drawn.  RQ3 explicitly promises "training
#          cost for the fine-tuned baseline" and "the cheapest freely available
#          option", neither of which the old output could support.
#
# [FIX-2]  RQ3 promises "model size" as part of the efficiency profile; no
#          parameter count existed anywhere in the pipeline.  Added as a
#          declared constant - it needs no re-inference.
#
# [FIX-3]  Hosted-API latency was contaminated by free-tier rate-limit waits:
#          Groq on fr_nfr had mean 2.070 s against a median of 0.094 s, a p95 of
#          0.719 s and a max of 59.194 s.  The mean measures the queue, not the
#          model.  Median is now reported alongside the mean, fig10 plots the
#          median, and a flag marks any cell where mean > 3x median.
#
# [FIX-4]  Training time is now split into the cost of ONE deployable model
#          (the longest single fold) and the cost of the WHOLE cross-validation
#          experiment.  Break-even uses the former; the latter belongs in the
#          reproducibility budget.
#
# [FIX-5]  cost3 now carries the encoder's evaluation regime, so an in-domain
#          encoder is never silently compared against a transfer number.
#
# [FIX-6]  macro-F1 is now computed over the same fixed label sets Stage 4 uses,
#          so a cost table can never disagree with the accuracy table.
# =============================================================================

import glob
import json
import os
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore")
OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)

# ----------------------------------------------------------------- price model
# AWS EC2 g4dn.xlarge (1x NVIDIA T4 16 GB), us-east-1, verified 2026-08-06.
# This is an IMPUTED rental price for the hardware, not a market price for the
# model, and it assumes sequential unbatched inference - both belong in RQ3's
# threats to validity. Batched serving would cut the local figures by roughly
# an order of magnitude, which is why the scaling table below is reported.
GPU_ON_DEMAND = 0.5260      # USD / hour
GPU_SPOT = 0.3162           # USD / hour
BATCH_SPEEDUPS = [1, 8, 32]  # sensitivity on the unbatched assumption
DEPLOY_SIZES = [1_000, 10_000, 100_000, 1_000_000]

# ------------------------------------------------------------------------ [FIX-2]
# Model size, in billions of parameters. RQ3 lists it as part of the efficiency
# profile. These are the published parameter counts of the exact checkpoints in
# stage2/stage3, declared here because they cannot be read back from a stored
# prediction. Gemini is closed-weight and Google publishes no count, so it is
# recorded as unknown rather than guessed.
PARAMS_B = {
    "bert-base-uncased-weighted": 0.110, "bert-base-uncased-unweighted": 0.110,
    "roberta-base-weighted": 0.125, "roberta-base-unweighted": 0.125,
    "gemma-2-2b-it": 2.61, "qwen2.5-3b-instruct": 3.09,
    "smollm3-3b": 3.08, "phi-4-mini-instruct": 3.84,
    "qwen2.5-7b-instruct": 7.62, "llama-3.1-8b-instruct": 8.03,
    "groq-llama-3.3-70b-versatile": 70.6,
    "gemini-3.1-flash-lite": np.nan,          # closed weights, size undisclosed
}

# ------------------------------------------------------------------------ [FIX-6]
# The same label sets Stage 4 scores against. Deriving labels from the observed
# y_true instead lets a cell that never saw a rare class score a higher macro-F1
# here than in tab1, and the two chapters would then disagree.
CATEGORIES_ALL = ["availability", "fault_tolerance", "legal", "look_and_feel",
                  "maintainability", "operational", "performance", "portability",
                  "scalability", "security", "usability"]
TOP4 = ["security", "usability", "operational", "performance"]
TOP6 = TOP4 + ["look_and_feel", "availability"]
LABELSETS = {
    "fr_nfr": ["FR", "NFR"], "security": ["security", "non-security"],
    "subtype_all": CATEGORIES_ALL, "subtype_top6": TOP6, "subtype_top4": TOP4,
}

TIER_COLOR = {"finetuned": "#0F766E", "open_local": "#2563EB",
              "open_hosted": "#7C3AED", "commercial": "#C026D3"}
TIER_LABEL = {"finetuned": "Fine-tuned encoder",
              "open_local": "LLM - local open-weight",
              "open_hosted": "LLM - hosted open-weight (API)",
              "commercial": "LLM - commercial closed-weight (API)"}
PRETTY = {
    "bert-base-uncased-weighted": "BERT (weighted)",
    "bert-base-uncased-unweighted": "BERT (unweighted)",
    "roberta-base-weighted": "RoBERTa (weighted)",
    "roberta-base-unweighted": "RoBERTa (unweighted)",
    "qwen2.5-7b-instruct": "Qwen2.5-7B", "qwen2.5-3b-instruct": "Qwen2.5-3B",
    "llama-3.1-8b-instruct": "Llama-3.1-8B", "phi-4-mini-instruct": "Phi-4-mini",
    "gemma-2-2b-it": "Gemma-2-2B", "smollm3-3b": "SmolLM3-3B",
    "groq-llama-3.3-70b-versatile": "Llama-3.3-70B (Groq)",
    "gemini-3.1-flash-lite": "Gemini-3.1-Flash-Lite",
}
pretty = lambda t: PRETTY.get(t, str(t).replace("openrouter-", "")
                              .replace("-instruct", "").replace("-", " ").title())


def find(*pats):
    for p in pats:
        h = sorted(glob.glob(p, recursive=True))
        if h:
            return h[0]
    return None


def read_any(p):
    if p is None:
        return None
    return (pd.read_parquet(p) if p.endswith(".parquet")
            else pd.read_csv(p, keep_default_na=False, low_memory=False))


frames, prov = [], {}
for name, pats in {
    "finetuned": ("/kaggle/input/**/predictions_finetuned.parquet",
                  "/kaggle/input/**/predictions_finetuned.csv"),
    # Repaired store first - it carries the positional parse and y_pred_strict,
    # and Stage 4 reads the same file. Two stages reading different versions of
    # one store is how a cost table stops reconciling with a results table.
    "llm_binary": ("/kaggle/input/**/predictions_llm_repaired.parquet",
                   "/kaggle/input/**/predictions_llm_repaired.csv",
                   "/kaggle/input/**/predictions_llm.parquet",
                   "/kaggle/input/**/predictions_llm.csv"),
    # Repaired store first. The previous order put a `_topup` file ahead of it,
    # and no notebook in this project produces one, so that pattern could only
    # ever miss - while the raw store sat ahead of the repaired .csv fallback.
    "llm_subtype": ("/kaggle/input/**/predictions_subtype_repaired.parquet",
                    "/kaggle/input/**/predictions_subtype_repaired.csv",
                    "/kaggle/input/**/predictions_subtype.parquet",
                    "/kaggle/input/**/predictions_subtype.csv"),
}.items():
    p = find(*pats)
    if p is None:
        print(f"[warn] {name}: not found")
        continue
    d = read_any(p)
    frames.append(d)
    prov[name] = {"path": p, "rows": int(len(d))}
    print(f"[load] {name:12s} {len(d):>7,} rows")
if not frames:
    raise FileNotFoundError("No prediction stores found. Check notebook Inputs.")

store = pd.concat(frames, ignore_index=True, sort=False)
for c in ["model_tag", "task", "dataset", "eval_regime", "approach", "prompt_id",
          "cost_basis", "y_true", "y_pred", "model_type"]:
    store[c] = store[c].astype(str)
store["shot_k"] = pd.to_numeric(store.get("shot_k", 0), errors="coerce").fillna(0).astype(int)
if "parse_ok" not in store.columns:
    store["parse_ok"] = True
store["parse_ok"] = store.parse_ok.astype(str).str.lower().isin(["true", "1"])
store.loc[store.approach == "finetuned", "parse_ok"] = True
for c in ["latency_s", "train_time_s", "cost_usd", "prompt_tokens", "completion_tokens",
          "price_in_per_mtok", "price_out_per_mtok"]:
    store[c] = pd.to_numeric(store.get(c, 0), errors="coerce").fillna(0.0)
store["model"] = store.model_tag.map(pretty)
store["tier"] = np.where(store.approach == "finetuned", "finetuned", store.model_type)
# Prefer the repaired store's y_pred_strict where it exists, exactly as Stage 4
# now does, so the two notebooks score identical rows identically.
_yhat = store.y_pred.where(store.parse_ok, "__unparsed__")
if "y_pred_strict" in store.columns:
    # notna() first - see the matching note in Stage 4. Coercing to str before
    # the NA test adopts an empty label on every row that lacks the column.
    _present = store["y_pred_strict"].notna()
    _sv = store["y_pred_strict"].where(_present, "").astype(str)
    _has = (_present & _sv.str.strip().ne("")
            & _sv.str.lower().ne("nan")).fillna(False).astype(bool)
    _yhat = _yhat.mask(_has, _sv)
store["y_hat"] = _yhat
store["correct"] = (store.y_hat == store.y_true)

# ---------------------------------------------------------------------- [FIX-1]
# Headline conditions: base prompt and zero-shot for the LLMs, every row for the
# encoders. Fine-tuned runs carry shot_k = -1 and prompt_id = "supervised" and
# must not be filtered on either.
IS_ENCODER = store.approach.astype(str).str.lower() == "finetuned"
IS_HEADLINE_PROMPT = (store.prompt_id.isin(["base", "nan", "", "none"])
                      & (store.shot_k == 0))
head = store[IS_ENCODER | IS_HEADLINE_PROMPT].copy()

_n_enc = int((head.approach.astype(str).str.lower() == "finetuned").sum())
print(f"[head] {len(head):,} rows | encoder {_n_enc:,} | prompted {len(head) - _n_enc:,}")
if _n_enc == 0:
    raise RuntimeError(
        "head contains ZERO fine-tuned encoder rows, so training cost and the "
        "break-even analysis cannot be computed and RQ3 is unanswerable. Check "
        "that predictions_finetuned is attached as a notebook input.")

# reportability gate from Stage 4; without it a cost-per-correct figure can be
# computed for a cell whose accuracy was never measurable in the first place.
gate_path = find("/kaggle/input/**/tab9_evaluability.csv", f"{OUT}/tab9_evaluability.csv")
if gate_path:
    gate = pd.read_csv(gate_path, keep_default_na=False)
    ok = set(zip(gate[gate.reportable.astype(str).str.lower().isin(["true", "1"])].model,
                 gate[gate.reportable.astype(str).str.lower().isin(["true", "1"])].task,
                 gate[gate.reportable.astype(str).str.lower().isin(["true", "1"])].dataset,
                 gate[gate.reportable.astype(str).str.lower().isin(["true", "1"])].eval_regime))
    _gate_enc = sum(1 for m, t, d, r in ok
                    if r in ("in_domain", "cross_project", "cross_dataset"))
    print(f"[gate] {gate_path} -> {len(ok)} reportable cells "
          f"({_gate_enc} of them encoder cells)")
    if _gate_enc == 0:
        raise RuntimeError(
            "The reportability gate contains no encoder cell. You are pointing "
            "at the OLD Stage 4 output. Re-run Stage 4 with [FIX-1] applied, "
            "commit it, then attach that new version as the input here.")
else:
    ok = None
    print("[gate] tab9_evaluability.csv not found - no reportability filter applied")


# ============================================================================
# TABLE 1 - unit economics per model x task x dataset
# ============================================================================
rows = []
for (m, tag, tier, task, ds, reg), g in head.groupby(
        ["model", "model_tag", "tier", "task", "dataset", "eval_regime"]):
    if ok is not None and (m, task, ds, reg) not in ok:
        continue
    n = len(g)
    n_corr = int(g.correct.sum())
    infer_s = float(g.latency_s.sum())
    api_usd = float(g.cost_usd.sum()) if g.cost_basis.iloc[0] == "api_tokens" else 0.0

    # -------------------------------------------------------------- [FIX-4]
    # Two different training costs. Deploying the classifier means training ONE
    # model, so the longest single fold is the fixed cost that break-even has to
    # repay. The sum across folds is the cost of the cross-validation
    # experiment, which a practitioner never pays.
    if "fold" in g.columns and g.fold.notna().any():
        per_fold = g.groupby("fold").train_time_s.first()
        train_one_s = float(per_fold.max()) if len(per_fold) else 0.0
        train_all_s = float(per_fold.sum()) if len(per_fold) else 0.0
    else:
        train_one_s = float(g.train_time_s.max())
        train_all_s = train_one_s

    if tier == "finetuned" or tier == "open_local":
        # local compute: imputed T4 rental for inference; training amortised
        # separately in the break-even table, never folded into a per-item rate
        infer_usd = infer_s / 3600.0 * GPU_ON_DEMAND
        train_usd = train_one_s / 3600.0 * GPU_ON_DEMAND
        exp_usd = train_all_s / 3600.0 * GPU_ON_DEMAND
        basis = "imputed T4 rental"
    else:
        infer_usd, train_usd, exp_usd, basis = api_usd, 0.0, 0.0, "API tokens"

    # -------------------------------------------------------------- [FIX-3]
    # Mean latency on a rate-limited free API measures the provider's queue, not
    # the model. Report the median too and flag the discrepancy explicitly.
    lat_mean = infer_s / n
    lat_med = float(g.latency_s.median())
    inflated = bool(lat_med > 0 and lat_mean > 3.0 * lat_med)

    rows.append({
        "model": m, "model_tag": tag, "tier": tier,
        "params_b": PARAMS_B.get(tag, np.nan),                       # [FIX-2]
        "task": task, "dataset": ds,
        "eval_regime": reg, "n": n, "accuracy": round(n_corr / n, 4),
        "cost_basis": basis,
        "latency_s_mean": round(lat_mean, 4),
        "latency_s_median": round(lat_med, 4),                       # [FIX-3]
        "latency_s_p95": round(float(g.latency_s.quantile(0.95)), 4),
        "latency_s_max": round(float(g.latency_s.max()), 4),
        "latency_inflated_by_queueing": inflated,                    # [FIX-3]
        "throughput_req_per_hour": round(3600.0 / max(lat_med, 1e-9), 1),
        "tokens_in": int(g.prompt_tokens.sum()), "tokens_out": int(g.completion_tokens.sum()),
        "train_time_s_one_model": round(train_one_s, 1),             # [FIX-4]
        "train_time_s_all_folds": round(train_all_s, 1),             # [FIX-4]
        "one_off_training_usd": round(train_usd, 4),
        "experiment_training_usd": round(exp_usd, 4),                # [FIX-4]
        "inference_usd_per_1k": round(infer_usd / n * 1000, 4),
        "inference_usd_per_1k_correct": (round(infer_usd / n_corr * 1000, 4)
                                         if n_corr else np.nan),
    })
cost1 = pd.DataFrame(rows).sort_values(["task", "dataset", "inference_usd_per_1k"])
cost1.to_csv(f"{OUT}/cost1_per_model.csv", index=False)
print(f"[cost1] {len(cost1)} rows")


# ============================================================================
# TABLE 2 - Pareto frontier (maximise macro-F1, minimise cost)
# ============================================================================
def macro_f1(g, task=None):
    # [FIX-6] fixed label set where the task is known, so this figure always
    # equals the one Stage 4 reports for the same cell.
    labs = LABELSETS.get(str(task)) or sorted(set(g.y_true))
    yp = g.y_hat if "y_hat" in g.columns else g.y_pred.where(g.parse_ok, "__unparsed__")
    f = []
    for c in labs:
        tp = int(((g.y_true == c) & (yp == c)).sum())
        fp = int(((g.y_true != c) & (yp == c)).sum())
        fn = int(((g.y_true == c) & (yp != c)).sum())
        f.append(0.0 if 2 * tp + fp + fn == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f))


f1map = {}
for (m, task, ds, reg), g in head.groupby(["model", "task", "dataset", "eval_regime"]):
    f1map[(m, task, ds, reg)] = round(macro_f1(g, task), 4)
cost1["macro_f1"] = [f1map.get((r.model, r.task, r.dataset, r.eval_regime), np.nan)
                     for r in cost1.itertuples()]


def pareto(df, x="inference_usd_per_1k", y="macro_f1"):
    """A point is on the frontier if nothing is both cheaper and better."""
    d = df.dropna(subset=[x, y]).sort_values([x, y], ascending=[True, False])
    keep, best = [], -np.inf
    for r in d.itertuples():
        v = getattr(r, y)
        if v > best:
            keep.append(r.Index)
            best = v
    return df.index.isin(keep)


cost1["on_pareto"] = False
for (task, ds), g in cost1.groupby(["task", "dataset"]):
    cost1.loc[g.index, "on_pareto"] = pareto(g)
cost1.to_csv(f"{OUT}/cost1_per_model.csv", index=False)
cost1[cost1.on_pareto][
    ["task", "dataset", "model", "tier", "macro_f1", "inference_usd_per_1k",
     "one_off_training_usd", "latency_s_mean"]
].sort_values(["task", "dataset", "macro_f1"], ascending=[True, True, False]).to_csv(
    f"{OUT}/cost2_pareto.csv", index=False)
print(f"[cost2] {int(cost1.on_pareto.sum())} Pareto-optimal configurations")


# ============================================================================
# TABLE 3 - break-even
#
# A fine-tuned encoder carries a one-off training cost and a very low marginal
# cost; an API model carries no fixed cost and a fixed marginal cost. The
# question RQ3 actually asks is at what deployment volume the encoder becomes
# cheaper - a single "cost per item" number cannot express that.
# ============================================================================
be = []
for (task, ds), g in cost1.groupby(["task", "dataset"]):
    fts = g[g.tier == "finetuned"]
    others = g[g.tier != "finetuned"]
    for f in fts.itertuples():
        for o in others.itertuples():
            m_f = f.inference_usd_per_1k / 1000.0
            m_o = o.inference_usd_per_1k / 1000.0
            fixed = f.one_off_training_usd
            n_star = fixed / (m_o - m_f) if m_o > m_f else np.inf
            be.append({
                "task": task, "dataset": ds,
                "encoder": f.model, "encoder_regime": f.eval_regime,   # [FIX-5]
                "encoder_f1": f.macro_f1,
                "alternative": o.model, "alternative_tier": o.tier,
                "alternative_f1": o.macro_f1,
                "f1_advantage_encoder": (round(f.macro_f1 - o.macro_f1, 4)
                                         if pd.notna(f.macro_f1) and pd.notna(o.macro_f1)
                                         else np.nan),
                "encoder_fixed_usd": round(fixed, 4),
                "encoder_marginal_usd_per_item": round(m_f, 8),
                "alternative_marginal_usd_per_item": round(m_o, 8),
                "breakeven_n_items": (round(n_star) if np.isfinite(n_star) else np.inf),
                "encoder_cheaper_at_1k": bool(fixed + 1_000 * m_f < 1_000 * m_o),
                "encoder_cheaper_at_100k": bool(fixed + 100_000 * m_f < 100_000 * m_o),
            })
cost3 = pd.DataFrame(be)
cost3.to_csv(f"{OUT}/cost3_breakeven.csv", index=False)
if not len(cost3):
    raise RuntimeError(
        "cost3 is empty: no encoder survived into cost1, so RQ3's break-even "
        "question cannot be answered. This is the [FIX-1] defect - check the "
        "head filter and the Stage 4 reportability gate.")
print(f"[cost3] {len(cost3)} encoder/alternative pairs | "
      f"finite break-even in {int(np.isfinite(cost3.breakeven_n_items.replace(np.inf, np.nan)).sum())}")


# ============================================================================
# TABLE 4 - sensitivity of the local-compute assumption
# ============================================================================
sens = []
for r in cost1.itertuples():
    for rate_name, rate in (("on_demand", GPU_ON_DEMAND), ("spot", GPU_SPOT)):
        for b in BATCH_SPEEDUPS:
            if r.tier in ("finetuned", "open_local"):
                v = r.inference_usd_per_1k * (rate / GPU_ON_DEMAND) / b
            else:
                v = r.inference_usd_per_1k       # API prices do not batch away
            sens.append({"model": r.model, "tier": r.tier, "task": r.task,
                         "dataset": r.dataset, "gpu_rate": rate_name,
                         "batch_speedup": b, "usd_per_1k": round(v, 5)})
cost4 = pd.DataFrame(sens)
cost4.to_csv(f"{OUT}/cost4_sensitivity.csv", index=False)


# ============================================================================
# TABLE 5 - total cost at realistic deployment sizes
# ============================================================================
scal = []
for r in cost1.itertuples():
    for n in DEPLOY_SIZES:
        total = r.one_off_training_usd + n * r.inference_usd_per_1k / 1000.0
        scal.append({"model": r.model, "tier": r.tier, "task": r.task,
                     "dataset": r.dataset, "macro_f1": r.macro_f1,
                     "n_requirements": n, "total_usd": round(total, 4),
                     "usd_per_1k_all_in": round(total / n * 1000, 4),
                     "hours_wall_clock": round(n * r.latency_s_mean / 3600.0, 2)})
cost5 = pd.DataFrame(scal)
cost5.to_csv(f"{OUT}/cost5_scaling.csv", index=False)


# ============================================================================
# FIGURES
# ============================================================================
plt.rcParams.update({"font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 160,
                     "savefig.bbox": "tight", "axes.grid": True,
                     "grid.alpha": 0.25, "grid.linestyle": ":"})


def save(fig, name):
    for e in ("png", "pdf"):
        fig.savefig(f"{OUT}/{name}.{e}")
    plt.close(fig)
    print(f"[fig] {name}")


def tier_legend(fig, extra=None):
    h = [Line2D([0], [0], marker="o", ls="", ms=9, color=c, label=TIER_LABEL[t])
         for t, c in TIER_COLOR.items()]
    if extra:
        h += extra
    fig.legend(handles=h, loc="lower center", ncol=min(5, len(h)),
               frameon=False, bbox_to_anchor=(0.5, -0.05))


# ------------------------------------------------------------------------ [FIX-9]
# An encoder now appears once per evaluation regime (in_domain plus
# cross_project or cross_dataset). The TABLES want all of them - that is what
# RQ2 is measured from. The COST FIGURES do not: plotting both draws the same
# model twice with no way to tell which point is which, which produced duplicate
# legend entries in fig8 and duplicate bars in fig9.
#
# RQ3 asks what a practitioner should deploy on their own data, and they would
# fine-tune on that data, so the deployment view keeps the in-domain row for
# each encoder. LLM rows are untouched - a prompted model has one regime only.
# The full per-regime detail stays in cost1_per_model.csv.
_enc = cost1.tier == "finetuned"
PLOT1 = cost1[(~_enc) | (cost1.eval_regime == "in_domain")].copy()
_dropped = int(len(cost1) - len(PLOT1))
print(f"[fig] deployment view for figures: {len(PLOT1)} rows "
      f"({_dropped} transfer-regime encoder rows kept in the tables, "
      f"not plotted)")
_dup = PLOT1.duplicated(["task", "dataset", "model"]).sum()
if _dup:
    raise RuntimeError(f"{_dup} models still appear twice in one panel; "
                       "the figures would be ambiguous. Check [FIX-9].")


# --- fig7: accuracy vs cost with the Pareto frontier ---------------------------
cells = [(t, d) for (t, d), g in PLOT1.groupby(["task", "dataset"])
         if g.macro_f1.notna().sum() >= 3]
# No silent caps: three panels keep the figure legible, but whatever is left
# out must be named, or the figure reads as if it covered everything.
if len(cells) > 3:
    print(f"[fig7] plotting the first 3 of {len(cells)} eligible cells; "
          f"omitted: {cells[3:]} - their numbers remain in cost1/cost2.")
cells = cells[:3]
if cells:
    fig, axes = plt.subplots(1, len(cells), figsize=(6.6 * len(cells), 5.8), squeeze=False,
                             gridspec_kw={"wspace": 0.30})
    for ax, (t, ds) in zip(axes[0], cells):
        g = PLOT1[(PLOT1.task == t) & (PLOT1.dataset == ds)].dropna(subset=["macro_f1"])
        # a free local model would vanish on a log axis; floor it at a visible epsilon
        xs = g.inference_usd_per_1k.clip(lower=1e-4)
        ax.scatter(xs, g.macro_f1, s=110,
                   c=[TIER_COLOR.get(x, "#888") for x in g.tier],
                   edgecolor="white", lw=1.2, zorder=3)
        fr = g[g.on_pareto].sort_values("inference_usd_per_1k")
        if len(fr) > 1:
            ax.step(fr.inference_usd_per_1k.clip(lower=1e-4), fr.macro_f1,
                    where="post", color="#111827", lw=1.4, ls="--", zorder=2,
                    label="Pareto frontier")
        for r in g.itertuples():
            ax.annotate(r.model, (max(r.inference_usd_per_1k, 1e-4), r.macro_f1),
                        textcoords="offset points", xytext=(7, 5), fontsize=7.5)
        ax.set_xscale("log")
        ax.set_xlabel("inference cost, USD per 1,000 requirements (log)")
        ax.set_ylabel("macro-F1")
        ax.set_title(f"{t} - {ds.upper()}", fontsize=11, weight="bold")
        if len(fr) > 1:
            ax.legend(fontsize=8, frameon=False, loc="lower right")
    fig.suptitle("RQ3 - Accuracy against inference cost", fontsize=14,
                 weight="bold", y=1.02)
    fig.text(0.5, -0.11, "Local models priced as imputed AWS T4 rental "
             f"(${GPU_ON_DEMAND}/hr, sequential unbatched); API models at "
             "published token rates.\nOne-off training cost is excluded here and "
             "handled in the break-even analysis. Encoders are shown at their "
             "in-domain accuracy - what a practitioner\nfine-tuning on their own "
             "data would face; their transfer numbers are in Table 2 and Figure 3.",
             ha="center", va="top", fontsize=9, color="#4B5563")
    tier_legend(fig)
    save(fig, "fig7_pareto")

# --- fig8: break-even curves ---------------------------------------------------
if len(cost3):
    pick = cost3[np.isfinite(cost3.breakeven_n_items.replace(np.inf, np.nan))]
    if len(pick):
        t, ds = pick.iloc[0][["task", "dataset"]]
        sub = pick[(pick.task == t) & (pick.dataset == ds)]
        enc = sub.encoder.iloc[0]
        # [FIX-9] pin ONE encoder and ONE regime, otherwise every alternative is
        # drawn twice and the legend lists each model two or three times.
        reg = ("in_domain" if (sub.encoder_regime == "in_domain").any()
               else sub.encoder_regime.iloc[0])
        sub = sub[(sub.encoder == enc) & (sub.encoder_regime == reg)]
        sub = sub.drop_duplicates("alternative").sort_values("breakeven_n_items")
        ns = np.logspace(2, 6, 200)
        fig, ax = plt.subplots(figsize=(10, 5.6))
        fixed = sub.encoder_fixed_usd.iloc[0]
        mf = sub.encoder_marginal_usd_per_item.iloc[0]
        ax.plot(ns, fixed + ns * mf, lw=2.4, color=TIER_COLOR["finetuned"],
                label=f"{enc}, {reg} (fine-tuned, ${fixed:.4f} one-off)")
        for r in sub.itertuples():
            ax.plot(ns, ns * r.alternative_marginal_usd_per_item, lw=1.7,
                    color=TIER_COLOR.get(r.alternative_tier, "#888"), alpha=0.85,
                    label=f"{r.alternative}")
        ax.set_xscale("log")
        ax.set_yscale("log")
        # annotations staggered across the vertical range so they never pile up
        _ylo, _yhi = ax.get_ylim()
        for j, r in enumerate(sub.itertuples()):
            if np.isfinite(r.breakeven_n_items) and 100 < r.breakeven_n_items < 1e6:
                ax.axvline(r.breakeven_n_items, color="#9CA3AF", lw=0.8, ls=":")
                ax.annotate(f"{int(r.breakeven_n_items):,}",
                            (r.breakeven_n_items,
                             _ylo * (_yhi / _ylo) ** (0.58 + 0.045 * (j % 8))),
                            rotation=90, fontsize=7.5, color="#4B5563",
                            ha="right", va="bottom")
        ax.set_xlabel("requirements classified (log)")
        ax.set_ylabel("cumulative cost, USD (log)")
        ax.set_title(f"Break-even: fine-tuning vs prompting - {t} / {ds.upper()}",
                     fontsize=13, weight="bold")
        ax.legend(fontsize=8, frameon=False, loc="lower right")
        fig.text(0.5, -0.02, "Vertical lines mark the deployment volume at which "
                 "the encoder's one-off training cost is repaid. Left of the line, "
                 "prompting is cheaper; right of it, fine-tuning is.",
                 ha="center", fontsize=9, color="#4B5563")
        save(fig, "fig8_breakeven")

# --- fig9: cost per 1,000 CORRECT classifications ------------------------------
g = PLOT1.dropna(subset=["inference_usd_per_1k_correct"])
if len(cells) and len(g):
    t, ds = cells[0]
    d = g[(g.task == t) & (g.dataset == ds)].sort_values("inference_usd_per_1k_correct")
    if len(d):
        fig, ax = plt.subplots(figsize=(10, max(4, 0.46 * len(d))))
        y = np.arange(len(d))
        vals = d.inference_usd_per_1k_correct.clip(lower=1e-5)
        ax.barh(y, vals, color=[TIER_COLOR.get(x, "#888") for x in d.tier], height=0.65)
        ax.set_yticks(y); ax.set_yticklabels(d.model, fontsize=9)
        ax.set_xscale("log")
        for i, v in enumerate(vals):
            ax.text(v * 1.12, i, f"${v:.4f}", va="center", fontsize=8)
        ax.set_xlabel("USD per 1,000 CORRECT classifications (log)")
        ax.set_title(f"Cost of a correct answer - {t} / {ds.upper()}",
                     fontsize=13, weight="bold")
        tier_legend(fig)
        save(fig, "fig9_cost_per_correct")

# --- fig10: latency ------------------------------------------------- [FIX-3] --
# Median, not mean. On the free API tiers a handful of rate-limit waits (Groq:
# max 59.2 s against a median of 0.094 s) drag the mean far above the p95, so a
# mean-based chart compares the provider's queue rather than the models.
lat = PLOT1.groupby(["model", "tier"]).agg(
    latency_s_median=("latency_s_median", "median"),
    latency_s_mean=("latency_s_mean", "mean"),
    queued=("latency_inflated_by_queueing", "any")).reset_index()
lat = lat.sort_values("latency_s_median")
if len(lat):
    fig, ax = plt.subplots(figsize=(10, max(4, 0.46 * len(lat))))
    y = np.arange(len(lat))
    ax.barh(y, lat.latency_s_median,
            color=[TIER_COLOR.get(x, "#888") for x in lat.tier], height=0.65)
    ax.set_yticks(y)
    ax.set_yticklabels([f"{m} *" if q else m
                        for m, q in zip(lat.model, lat.queued)], fontsize=9)
    for i, (v, mu) in enumerate(zip(lat.latency_s_median, lat.latency_s_mean)):
        ax.text(v * 1.02, i, f"{v:.3f}s  (mean {mu:.3f}s)", va="center", fontsize=8)
    ax.set_xlabel("median seconds per requirement")
    ax.set_title("Latency per classification (median)", fontsize=13, weight="bold")
    fig.text(0.5, -0.05, "Local models: sequential unbatched inference on one T4. "
             "API models include network round-trip. An asterisk marks a model "
             "whose mean exceeds 3x its median because free-tier rate limiting "
             "inserted retry waits - for those, the mean is not a model property.",
             ha="center", fontsize=9, color="#4B5563")
    tier_legend(fig)
    save(fig, "fig10_latency")


# ============================================================================
json.dump({
    "sources": prov,
    "gpu_price_usd_per_hour": {"on_demand": GPU_ON_DEMAND, "spot": GPU_SPOT},
    "gpu_price_source": ("AWS EC2 g4dn.xlarge (1x T4), us-east-1, "
                         "verified 2026-08-06"),
    "assumptions": [
        "Local inference is priced as imputed hardware rental, not a market "
        "price for the model.",
        "Latency was measured under sequential unbatched inference; batching "
        "would reduce local cost by roughly the batch factor (see cost4).",
        "One-off training cost is reported separately and never amortised into "
        "a per-item rate; the break-even table is the amortisation.",
        "API prices are published list rates at the date above and exclude "
        "free-tier allowances.",
        "one_off_training_usd prices ONE deployable model (the longest single "
        "fold). experiment_training_usd prices the whole cross-validation run.",
        "Latency is reported as a median; on rate-limited free API tiers the "
        "mean measures provider queueing and is flagged, not used.",
        "params_b is the published parameter count of the checkpoint; Gemini is "
        "closed-weight and Google discloses no figure, so it is null.",
    ],
    "reportability_gate": gate_path,
    "n_rows": int(len(store)),
    "head_encoder_rows": _n_enc,
    "cost1_rows": int(len(cost1)),
    "cost1_encoder_rows": int((cost1.tier == "finetuned").sum()),
    "breakeven_pairs": int(len(cost3)),
    "latency_cells_flagged_as_queued": int(cost1.latency_inflated_by_queueing.sum()),
    "pareto_optimal": int(cost1.on_pareto.sum()),
}, open(f"{OUT}/stage5_provenance.json", "w"), indent=2)

print("\n[done] outputs:")
for f in sorted(glob.glob(f"{OUT}/cost*") + glob.glob(f"{OUT}/fig7*") +
                glob.glob(f"{OUT}/fig8*") + glob.glob(f"{OUT}/fig9*") +
                glob.glob(f"{OUT}/fig10*") + glob.glob(f"{OUT}/stage5_provenance.json")):
    print(f"   {os.path.basename(f):36s} {os.path.getsize(f)/1024:8.1f} KB")


ATTACHED INPUTS

unified.parquet   <- Stage 1 corpus        (Stages 2, 4)
   (not attached)

splits.json   <- Stage 1 splits        (Stage 2)
   (not attached)

predictions_finetuned.parquet   <- Stage 2 encoder store (Stage 2 resume, 4, 5)
   /kaggle/input/datasets/zahrasamir/stage2-outputs/predictions_finetuned.parquet
           210,255 bytes   2026-08-27 08:58  >> THIS ONE WILL BE USED

predictions_llm.parquet   <- Stage 3 LLM binary    (Stages 4, 5)
   (not attached)

predictions_subtype.parquet   <- Stage 3b raw subtype  (Stage 3b-repair)
   (not attached)

predictions_subtype_repaired.parquet   <- Stage 3b repaired    (Stages 4, 5)
   /kaggle/input/notebooks/zahrasamir/stage3b-repair/predictions_subtype_repaired.parquet
           428,497 bytes   2026-08-27 08:58  >> THIS ONE WILL BE USED

predictions_llm_repaired.parquet   <- Stage 3b R8 binary   (Stages 4, 5)
   /kaggle/input/notebooks/zahrasamir/stage3b-repair/predictions_llm_repaired.parquet
           841,382 bytes   2026-0